# Lab 3: Automated Inference Recommendations

In this lab, you will use **SageMaker AI Inference Recommendations** to automatically find the optimal deployment configuration for Meta Llama 3.1 8B Instruct. Instead of manually deploying and benchmarking each configuration, the service will:

1. **Analyze** your model's architecture and memory requirements
2. **Narrow** the configuration space to viable options
3. **Benchmark** each configuration on real GPU infrastructure
4. **Return ranked recommendations** with validated performance metrics

---

### ⚠️ Resource Requirements

Recommendation jobs provision **real GPU instances** in your account to benchmark each candidate configuration. This means:
- **Cost**: Each instance type evaluated incurs compute charges for 10-30 minutes
- **Quota**: You need service quota for the instance types being evaluated (e.g., `ml.g5.12xlarge`, `ml.g6.12xlarge`)
- **Time**: Jobs typically take 30-90 minutes; enabling model optimization (speculative decoding) can extend this to 60-120+ minutes
- **IAM**: Your execution role needs `servicequotas:GetServiceQuota` permission

For a full optimization pass (speculative decoding, kernel tuning across many instance types), expect significant compute usage. This lab uses a focused configuration (3 instance types, no model optimization) to keep costs and time manageable.

---

## Benchmarking vs. Recommendations: Two Different Tools

| | Automated Benchmarking | Inference Recommendations |
|---|---|---|
| **Input** | An existing endpoint | Model artifacts in S3 |
| **What it does** | Measures performance of what you already deployed | Finds the BEST way to deploy |
| **Optimizations** | None - measures as-is | Can apply speculative decoding, kernel tuning |
| **Instance types** | Tests the one you deployed on | Compares across up to 3 instance types |
| **Output** | Performance metrics | Deployment-ready configurations + metrics |
| **Use case** | Validate current deployment | Pre-deployment optimization |

> 💡 **Key insight**: Use Benchmarking to measure what you HAVE. Use Recommendations to find what you SHOULD have.


## Setup

In [ ]:
%pip install -r ../requirements.txt --quiet --no-warn-conflicts

In [ ]:
import json
import os
import boto3
import pandas as pd
from datetime import datetime

import sys
sys.path.append("..")
from utils import (
    get_execution_role,
    wait_for_recommendation_job,
    format_recommendation,
)

role = get_execution_role()
region = boto3.session.Session().region_name or "us-west-2"
account_id = boto3.client("sts").get_caller_identity()["Account"]
bucket = f"sagemaker-{region}-{account_id}"
timestamp = datetime.now().strftime("%m%d%H%M%S")

sm_client = boto3.client("sagemaker", region_name=region)
s3_client = boto3.client("s3", region_name=region)

print(f"Region:  {region}")
print(f"Role:    {role}")
print(f"Bucket:  {bucket}")


## 1. Upload Model Artifacts to S3

Unlike benchmarking (which targets an existing endpoint), the Recommendations API takes model weights from S3 as input — it needs the artifacts to analyze the architecture and apply optimizations like speculative decoding.

We stream files directly from HuggingFace Hub to S3 without writing to local disk. If you've already run this lab before, the upload is skipped automatically.

> ⚠️ **Important**: You need a HuggingFace token with access to the gated Meta Llama 3.1 model. Request access at: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct


In [ ]:
model_name_hf = "meta-llama/Llama-3.1-8B-Instruct"
model_s3_prefix = "models/llama-3.1-8b-instruct"
model_s3_uri = f"s3://{bucket}/{model_s3_prefix}/"

HF_TOKEN = "YOUR_HF_TOKEN_HERE"  # Replace with your token

print(f"Model:          {model_name_hf}")
print(f"S3 destination: {model_s3_uri}")


In [ ]:
# Check if model is already in S3 — skip upload if so
response = s3_client.list_objects_v2(Bucket=bucket, Prefix=model_s3_prefix, MaxKeys=1)
already_uploaded = response.get("KeyCount", 0) > 0

if already_uploaded:
    print(f"✅ Model already in S3 at {model_s3_uri} — skipping upload")
else:
    print(f"Uploading {model_name_hf} to {model_s3_uri}...")
    import requests
    headers = {"Authorization": f"Bearer {HF_TOKEN}"}
    files = requests.get(
        f"https://huggingface.co/api/models/{model_name_hf}",
        headers=headers,
    ).json().get("siblings", [])

    skip = (".bin", ".pt", ".gguf")
    to_upload = [f["rfilename"] for f in files if not f["rfilename"].endswith(skip)]
    print(f"  {len(to_upload)} files to upload")

    for filename in to_upload:
        url = f"https://huggingface.co/{model_name_hf}/resolve/main/{filename}"
        s3_key = f"{model_s3_prefix}/{filename}"
        with requests.get(url, headers=headers, stream=True) as r:
            r.raise_for_status()
            s3_client.upload_fileobj(r.raw, bucket, s3_key)
        print(f"  ✅ {filename}")

    print(f"\nDone. Model available at: {model_s3_uri}")


In [ ]:
# Verify model files are in S3
response = s3_client.list_objects_v2(Bucket=bucket, Prefix=model_s3_prefix, MaxKeys=20)

if "Contents" in response:
    print(f"✅ {len(response['Contents'])} files found at {model_s3_uri}")
    for obj in response["Contents"][:8]:
        size_mb = obj["Size"] / (1024 * 1024)
        print(f"   {obj['Key'].split('/')[-1]:50s} {size_mb:8.1f} MB")
    if len(response["Contents"]) > 8:
        print(f"   ... and {len(response['Contents']) - 8} more files")
else:
    print(f"❌ No files found at {model_s3_uri}")
    print("   Run the upload cell above first, or set model_s3_uri to an existing location.")


## 2. Create a Workload Configuration

Same baseline workload parameters as Lab 1 for a direct comparison.


In [ ]:
workload_config_name = f"rec-workload-{timestamp}"

workload_spec = {
    "benchmark": {"type": "aiperf"},
    "parameters": {
        "prompt_input_tokens_mean": 550,
        "prompt_input_tokens_stddev": 150,
        "output_tokens_mean": 150,
        "output_tokens_stddev": 50,
        "concurrency": 10,
        "streaming": True,
    },
    "tooling": {"api_standard": "openai"},
}

response = sm_client.create_ai_workload_config(
    AIWorkloadConfigName=workload_config_name,
    AIWorkloadConfigs={"WorkloadSpec": {"Inline": json.dumps(workload_spec)}},
)

print(f"Workload config created: {workload_config_name}")
print(f"ARN: {response['AIWorkloadConfigArn']}")


## 3. Create the Recommendation Job

The recommendation job evaluates each candidate instance type and returns ranked, deployment-ready configurations with validated metrics.

> ⚠️ Recommendation jobs take **30-90 minutes** — they provision real GPU instances and benchmark each configuration.


In [ ]:
recommendation_job_name = f"rec-llama31-8b-{timestamp}"
s3_output_location = f"s3://{bucket}/recommendation-results/{recommendation_job_name}/"

response = sm_client.create_ai_recommendation_job(
    AIRecommendationJobName=recommendation_job_name,
    ModelSource={"S3": {"S3Uri": model_s3_uri}},
    OutputConfig={"S3OutputLocation": s3_output_location},
    PerformanceTarget={
        "Constraints": [{"Metric": "throughput"}]
    },
    AIWorkloadConfigIdentifier=workload_config_name,
    ComputeSpec={
        "InstanceTypes": [
            "ml.g5.12xlarge",   # 4x NVIDIA A10G (96GB total)
            "ml.g6.12xlarge",   # 4x NVIDIA L4 (96GB total)
            "ml.g6e.12xlarge",  # 4x NVIDIA L40S Enhanced
        ]
    },
    OptimizeModel=False,
    InferenceSpecification={"Framework": "LMI"},
    RoleArn=role,
)

recommendation_job_arn = response["AIRecommendationJobArn"]
print(f"Recommendation job created!")
print(f"  Name:   {recommendation_job_name}")
print(f"  ARN:    {recommendation_job_arn}")
print(f"  Output: {s3_output_location}")
print(f"\nOptimization target: THROUGHPUT (speculative decoding)")
print(f"Instance types:      g5.12xlarge | g6.12xlarge | g6e.12xlarge")
print(f"Estimated time:      30-90 minutes")


## 4. Monitor Progress


In [ ]:
job_response = wait_for_recommendation_job(
    job_name=recommendation_job_name,
    region=region,
    timeout_minutes=120,
)


## 5. Review Recommendations

Each recommendation includes:
- **DeploymentConfiguration**: Container image, instance type, environment variables
- **ExpectedPerformance**: Validated metrics from real benchmarking
- **OptimizationDetails**: Techniques applied (tensor parallelism, kernel tuning)


In [ ]:
response = sm_client.describe_ai_recommendation_job(
    AIRecommendationJobName=recommendation_job_name
)

recommendations = response.get("Recommendations", [])
print(f"Received {len(recommendations)} recommendations:\n")

for i, rec in enumerate(recommendations):
    print(format_recommendation(rec, index=i))


In [ ]:
# Build a comparison table
comparison_data = []

for i, rec in enumerate(recommendations):
    deploy_config = rec.get("DeploymentConfiguration", {})
    performance = rec.get("ExpectedPerformance", [])
    optimization = rec.get("OptimizationDetails", {})

    row = {
        "Rank": i + 1,
        "Instance Type": deploy_config.get("InstanceType", "N/A"),
        "Instance Count": deploy_config.get("InstanceCount", 1),
        "Copies/Instance": deploy_config.get("CopyCountPerInstance", 1),
    }
    for metric in performance:
        name = metric.get("Metric", metric.get("Name", ""))
        stat = metric.get("Stat", "")
        key = f"{name} ({stat})" if stat else name
        row[key] = metric.get("Value", "")
    if isinstance(optimization, dict):
        row["Optimizations"] = ", ".join(optimization.keys())
    elif isinstance(optimization, list):
        row["Optimizations"] = ", ".join(str(o) for o in optimization)

    comparison_data.append(row)

df = pd.DataFrame(comparison_data)
print("\n" + "=" * 80)
print("  RECOMMENDATION COMPARISON")
print("=" * 80)
print(df.to_string(index=False))


## 6. Understanding the Results

The recommendations compare deployment configurations across instance types. Key factors:

- **Tensor Parallelism**: Automatically distributes model layers across GPUs for optimal utilization
- **Instance Selection**: Different GPU types (A10G vs L4 vs L40S) trade off memory, compute, and cost
- **Throughput vs Latency**: The ranking depends on your performance target

> 💡 **Want speculative decoding?** Set `OptimizeModel=True` and provide a `DatasetConfig` with training data (ShareGPT format in S3). This trains an EAGLE speculator for 1.5-2.5x throughput improvement but requires broader instance quota and longer job time (60-120 min).


## 7. Save Results for Lab 4


In [ ]:
os.makedirs("../results", exist_ok=True)

recommendation_data = {
    "job_name": recommendation_job_name,
    "recommendations": recommendations,
    "model_s3_uri": model_s3_uri,
    "workload_config_name": workload_config_name,
    "timestamp": timestamp,
}

with open("../results/lab3_recommendations.json", "w") as f:
    json.dump(recommendation_data, f, indent=2, default=str)

print("Saved to: ../results/lab3_recommendations.json")
print("Lab 4 will load this to deploy the recommended configuration.")


## 8. Cleanup

> ⚠️ Do **not** delete `../results/lab3_recommendations.json` — Lab 4 needs it.

Temporary endpoints created by the recommendation job are cleaned up automatically by SageMaker AI.


In [ ]:
sm_client.delete_ai_workload_config(AIWorkloadConfigName=workload_config_name)
print(f"✅ Workload config '{workload_config_name}' deleted")
print("\nNote: Temporary recommendation endpoints are automatically cleaned up by SageMaker AI.")


---

## Summary

In this lab you:
- Uploaded model artifacts directly from HuggingFace Hub to S3 (no local disk)
- Created a recommendation job evaluating 3 instance types with automatic optimization
- Reviewed ranked, deployment-ready configurations with validated performance metrics
- Saved the best recommendation for deployment in Lab 4

**Next**: In [Lab 4](../lab4/lab4_deploy_and_compare.ipynb), you'll deploy the recommended configuration and compare it head-to-head against the Lab 1 baseline.